## Part 1: Preprocessing

In [ ]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

In [ ]:
# Determine the number of unique values in each column.
attrition_df.nunique().sort_values(ascending=False)

In [ ]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[["Attrition", "Department"]]

In [ ]:
# Create a list of at least 10 column names to use as X data
X_columns_to_include = [
    column_name
    for column_name in attrition_df.columns
    if column_name not in ("Attrition", "Department")
]

# Create X_df using your selected columns
X_df = attrition_df[X_columns_to_include]

# Show the data types for X_df
X_df.info()

In [ ]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.2, random_state=42)


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create an instance of the label encoder
le = LabelEncoder()

# List of categorical columns to be encoded
categorical_columns = [
    "OverTime",
    "BusinessTravel",
    "EducationField",
    "JobRole",
    "MaritalStatus",
]

# Fit the label encoder on the combined data from both training and test sets
for column in categorical_columns:
    combined_data = list(X_train[column]) + list(X_test[column])
    le.fit(combined_data)

    # Transform the training data
    X_train[column] = le.transform(X_train[column])

    # Transform the test data
    X_test[column] = le.transform(X_test[column])

# Print value counts for each encoded column in the training set
for column in categorical_columns:
    print(f"Value counts for {column} in training set:")
    print(X_train[column].value_counts())

# Print value counts for each encoded column in the test set
for column in categorical_columns:
    print(f"Value counts for {column} in test set:")
    print(X_test[column].value_counts())

In [ ]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
X_scaler = scaler.fit(X_train)

# Scale the training and testing data
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
pd.DataFrame(X_train_scaled, columns=X_train.columns).head()

In [ ]:
# Create a OneHotEncoder for the Department column
from sklearn.preprocessing import OneHotEncoder
dept_encoder = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
dept_encoder.fit(y_train['Department'].values.reshape(-1,1))

# Create two new variables by applying the encoder
# to the training and testing data
encoded_train_Department = dept_encoder.transform(y_train['Department'].values.reshape(-1,1))
encoded_test_Department = dept_encoder.transform(y_test['Department'].values.reshape(-1,1))

encoded_train_Department[:]



In [ ]:
# Create a OneHotEncoder for the Attrition column
attr_encoder = OneHotEncoder(sparse_output=False)


# Fit the encoder to the training data
attr_encoder.fit(y_train['Attrition'].values.reshape(-1,1))


# Create two new variables by applying the encoder
# to the training and testing data
encoded_train_Attrition = attr_encoder.transform(y_train['Attrition'].values.reshape(-1,1))
encoded_test_Attrition = attr_encoder.transform(y_test['Attrition'].values.reshape(-1,1))

encoded_train_Attrition[:]


## Create, Compile, and Train the Model

In [ ]:
# Find the number of columns in the X training data
input_features = X_train_scaled.shape[1]

# Create the input layer
input_layer = layers.Input(shape=(input_features,))

# Create at least two shared layers
shared_layer1 = layers.Dense(64, activation='relu', name='Shared_1')(input_layer)
shared_layer2 = layers.Dense(32128, activation='relu', name='Shared_2')(shared_layer1)


In [ ]:
# Create a branch for Department with a hidden layer and an output layer
# Create the hidden layer
branch_hidden_layer_1 = layers.Dense(32, activation='relu', name = 'hid_dept')(shared_layer2)

# Create the output layer
output_layer = layers.Dense(3, activation='softmax', name = 'out_dept')(branch_hidden_layer_1)



In [ ]:
# Create a branch for Attrition with a hidden layer and an output layer
# Create the hidden layer
branch_hidden_layer_2 = layers.Dense(32, activation='relu', name = 'hid_attrition')(shared_layer2)

# Create the output layer
output_layer2 = layers.Dense(2, activation='sigmoid', name = 'out_attrition')(branch_hidden_layer_2)


In [ ]:
# Create the model
model = Model(inputs=input_layer, outputs=[output_layer, output_layer2])


# Compile the model
model.compile(optimizer='adam',
              loss={'out_dept': 'categorical_crossentropy', 'out_attrition': 'categorical_crossentropy'},
              metrics={'out_dept':'accuracy', 'out_attrition':'accuracy'})

# Summarize the model
model.summary()

In [ ]:
# Train the model
model.fit(X_train_scaled, {'out_dept': encoded_train_Department, 'out_attrition': encoded_train_Attrition}, epochs=100, shuffle=True, verbose=2)


In [ ]:
# Evaluate the model with the testing data
results_test = model.evaluate(X_test_scaled, {'out_dept': encoded_test_Department, 'out_attrition': encoded_test_Attrition}, verbose=2)
print(results_test)


In [ ]:
# Print the accuracy for both department and attrition
print(f"Department predictions Accuracy: {results_test[2]}")
print(f"Attrition predictions Accuracy: {results_test[1]}")


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. The imbalance in our dataset, where certain classes dominate others (like "no attrition" and "Research & Development"), makes accuracy a deceptive metric. A simplistic model that always predicts these majority classes might appear accurate, yet it lacks practical value.

To truly gauge the effectiveness of our model, we must consider the specific business context. If our main objective is to identify employees likely to leave and implement retention strategies, then a model that minimizes false negatives (high recall) is crucial. This means prioritizing the correct identification of potential attrition, even if it leads to some false positives (incorrectly predicting attrition for some employees). 

Therefore, beyond accuracy, we'll focus on metrics like recall and precision, balancing their trade-off to ensure the model aligns with our business goals.

2. In my model's architecture, I employed different activation functions for the department and attrition output layers. Specifically, I chose the softmax function for the department output, as it adeptly distributes probabilities across multiple classes, ensuring their sum totals 1. This is ideal for multi-class classification tasks like predicting which department an employee might belong to.

Conversely, for the attrition output layer, I utilized the sigmoid function. This function is well-suited for binary classification, mapping input values to a range between 0 and 1, effectively representing probabilities of "yes/no" or "true/false" outcomes. In this context, the sigmoid function is ideal for predicting whether an employee will leave the company (attrition).

By strategically selecting activation functions that align with the nature of each output layer's task, I aimed to enhance the model's overall predictive performance and interpretability.


3. Several avenues exist for enhancing the model's performance:

* **Architecture Modifications:** Experimenting with the model's structure by adding more layers or neurons could increase its capacity to learn complex patterns.
* **Hyperparameter Optimization:** Fine-tuning hyperparameters such as learning rate, batch size, the number of epochs, and optimizer choice can significantly impact model performance.
* **Feature Engineering:** Creating new features or transforming existing ones could uncover hidden relationships in the data and improve predictive power.
* **Data Balancing:** Addressing biases or imbalances in the dataset, perhaps through techniques like oversampling or undersampling, can help the model generalize better.

These are just a few potential strategies for improving the model. A systematic approach involving experimentation and careful analysis will be crucial in determining the most effective enhancements.
